<a href="https://colab.research.google.com/github/AliRaddman/divar-ml-project/blob/main/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" align="right">

# نوت‌بوک آماده‌سازی و تمیزکاری داده‌ها

این نوت‌بوک نسخه‌ی نهایی و ادغام‌شده‌ی مرحله‌ی preprocessing پروژه است.

در این فایل، خروجی‌های کاری اعضای تیم از نوت‌بوک‌های جداگانه ادغام می‌شوند و دیتاست خام دیوار به یک نسخه‌ی تمیز و قابل استفاده برای تحلیل آماری، نقشه، خوشه‌بندی و مدل‌سازی تبدیل می‌شود.

مراحل اصلی این نوت‌بوک:

1. راه‌اندازی محیط و نصب کتابخانه‌ها
2. بارگذاری دیتای خام
3. اصلاح نوع داده‌ها و مدیریت مقادیر گم‌شده
4. ساخت ستون‌های مربوط به قیمت، متراژ و قیمت هدف
5. ساخت flagهای اعتبارسنجی برای تحلیل قیمت
6. پاکسازی داده‌های جغرافیایی
7. تبدیل مختصات جغرافیایی به UTM
8. ذخیره خروجی نهایی preprocessing

</div>

<div dir="rtl" align="right">

## 1. راه‌اندازی محیط و import کتابخانه‌ها

در این مرحله Google Drive به Colab متصل می‌شود و کتابخانه‌های موردنیاز برای preprocessing، کار با فایل‌های parquet، تبدیل تاریخ شمسی و تبدیل مختصات به UTM آماده می‌شوند.

</div>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q jdatetime utm pyarrow

import os
import pandas as pd
import numpy as np
import jdatetime
import utm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

print("Setup completed.")

Mounted at /content/drive
Setup completed.


<div dir="rtl" align="right">

## 2. بارگذاری دیتای خام

در این مرحله دیتاست خام آگهی‌های املاک دیوار از Google Drive خوانده می‌شود.

این فایل نقطه شروع preprocessing است و در مراحل بعدی، نوع داده‌ها، مقادیر گم‌شده، ستون‌های قیمت و مختصات جغرافیایی روی آن پردازش می‌شوند.

</div>

In [2]:
raw_data_path = "/content/drive/MyDrive/Divar Dataset/Divar.csv"

df = pd.read_csv(raw_data_path)

print("Shape:")
print(df.shape)

print("\nNumber of columns:")
print(len(df.columns))

df.head()

/tmp/ipykernel_5441/2399571182.py:3: DtypeWarning: Columns (11,27,29,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_data_path)


Shape:
(1000000, 61)

Number of columns:
61


,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,rent_to_single,rent_type,price_mode,price_value,credit_mode,credit_value,rent_credit_transform,transformable_price,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,deed_type,has_business_deed,floor,rooms_count,total_floors_count,unit_per_floor,has_balcony,has_elevator,has_warehouse,has_parking,construction_year,is_rebuilt,has_water,has_warm_water_provider,has_electricity,has_gas,has_heating_system,has_cooling_system,has_restroom,has_security_guard,has_barbecue,building_direction,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,NaN,NaN,NaN,سه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,NaN,NaN,مقطوع,8.500000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,3,یک,NaN,NaN,NaN,True,True,True,۱۳۸۴,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,NaN,NaN,NaN,NaN,مقطوع,750000000.0,False,False,750000000.0,NaN,26000000.0,NaN,NaN,132.0,NaN,NaN,3,سه,NaN,NaN,NaN,True,True,True,۱۴۰۱,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,NaN,NaN,NaN,NaN,مقطوع,950000000.0,False,False,950000000.0,NaN,95000000.0,NaN,NaN,90.0,NaN,NaN,4,یک,NaN,NaN,NaN,True,False,True,۱۴۰۰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,NaN,NaN,مقطوع,5.750000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,115.0,single_page,NaN,4,دو,6,NaN,true,True,True,True,۱۴۰۳,NaN,NaN,package,NaN,NaN,shoofaj,air_conditioner,squat_seat,NaN,NaN,north,NaN,NaN,NaN,ceramic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div dir="rtl" align="right">

## 3. اصلاح نوع داده‌ها و مدیریت مقادیر گم‌شده

در این بخش ستون‌های عددی، دسته‌ای، بولین و تاریخی اصلاح می‌شوند.

همچنین مقادیر گم‌شده بر اساس منطق هر ستون مدیریت می‌شوند.
برخی NaNها به دلیل ماهیت ساختاری ستون‌ها حفظ می‌شوند و برخی دیگر با روش‌هایی مثل میانه‌ی کل یا میانه‌ی هر دسته پر می‌شوند.

خروجی این مرحله:

`cleaned_step1.parquet`

</div>


<div dir="rtl" align="right">

### 3.1 اصلاح نوع داده‌ها

در این مرحله نوع داده‌ی ستون‌های اصلی اصلاح می‌شود.

ستون‌های عددی مثل `floor`، `total_floors_count`، `rooms_count`، `unit_per_floor` و `construction_year` به عدد تبدیل می‌شوند.

ستون‌های بولین مثل امکانات ملک و flagهای قیمتی به نوع `boolean` تبدیل می‌شوند.

همچنین مقدار `unselect` به `NaN` تبدیل می‌شود و ستون‌های دسته‌ای به نوع `category` تغییر می‌کنند.

</div>

In [3]:
# ستون‌های مربوط به طبقه را عددی می‌کنیم
for col in ["floor", "total_floors_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# تعداد اتاق‌ها در دیتا به‌صورت متنی آمده بود
rooms_map = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5,
}

df["rooms_count"] = df["rooms_count"].map(rooms_map)


# تعداد واحد در هر طبقه را عددی می‌کنیم
df["unit_per_floor"] = df["unit_per_floor"].replace("more_than_8", 9)
df["unit_per_floor"] = pd.to_numeric(df["unit_per_floor"], errors="coerce")


# ارقام فارسی سال ساخت را به انگلیسی تبدیل می‌کنیم
def fa_to_en_digits(text):
    if pd.isna(text):
        return text

    fa_digits = "۰۱۲۳۴۵۶۷۸۹"
    en_digits = "0123456789"
    translation_table = str.maketrans(fa_digits, en_digits)

    return str(text).translate(translation_table)


df["construction_year"] = df["construction_year"].apply(fa_to_en_digits)
df["construction_year"] = df["construction_year"].replace("قبل از 1370", 1365)
df["construction_year"] = pd.to_numeric(df["construction_year"], errors="coerce")


# این ستون‌ها واقعاً بولین هستند
clean_bool_cols = [
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna",
    "has_business_deed",
    "is_rebuilt",
    "transformable_price",
    "rent_credit_transform",
]

# بالکن به‌صورت متن ذخیره شده بود، اول مقدارهایش را درست می‌کنیم
df["has_balcony"] = df["has_balcony"].replace({
    "true": True,
    "false": False,
    "unselect": np.nan,
})

all_bool_cols = clean_bool_cols + ["has_balcony"]

for col in all_bool_cols:
    df[col] = df[col].astype("boolean")


# مقدار unselect یعنی انتخاب نشده، پس به NaN تبدیلش می‌کنیم
df = df.replace("unselect", np.nan)


# ستون‌های دسته‌ای را category می‌کنیم که هم سبک‌تر باشند هم معنی‌شان مشخص‌تر شود
category_cols = [
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "neighborhood_slug",
    "user_type",
    "rent_mode",
    "rent_type",
    "price_mode",
    "credit_mode",
    "deed_type",
    "building_direction",
    "floor_material",
    "property_type",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
]

for col in category_cols:
    df[col] = df[col].astype("category")


# این ستون فقط index قبلی فایل بوده و لازم نیست
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])


print("Step 3.1 completed.")
print("Shape:", df.shape)
print("\nDtype summary:")
print(df.dtypes.value_counts())

Step 3.1 completed.
Shape: (1000000, 60)

Dtype summary:
float64     22
boolean     13
object       8
category     3
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


<div dir="rtl" align="right">

### 3.2 تبدیل تاریخ و حذف ستون‌های غیرضروری

در این مرحله از ستون `created_at_month` دو ستون شمسی می‌سازیم.

ستون `created_at_shamsi` تاریخ را به فرم عددی شمسی نگه می‌دارد و ستون `created_at_shamsi_readable` نسخه‌ی خواناتر ماه و سال شمسی است.

همچنین چند ستون که برای ادامه تحلیل لازم نیستند یا داده‌ی قابل اتکایی ندارند، از دیتاست حذف می‌شوند.

</div>

In [ ]:
# تبدیل تاریخ میلادی ماهانه به تاریخ شمسی
def gregorian_month_to_shamsi(value):
    if pd.isna(value):
        return pd.NA

    date_value = pd.to_datetime(str(value) + "-01", errors="coerce")

    if pd.isna(date_value):
        return pd.NA

    shamsi_date = jdatetime.date.fromgregorian(date=date_value.date())
    return f"{shamsi_date.year:04d}-{shamsi_date.month:02d}"


# نسخه خواناتر تاریخ شمسی، برای گزارش و نمودار
def shamsi_month_readable(value):
    if pd.isna(value):
        return pd.NA

    year, month = str(value).split("-")
    month = int(month)

    month_names = {
        1: "فروردین",
        2: "اردیبهشت",
        3: "خرداد",
        4: "تیر",
        5: "مرداد",
        6: "شهریور",
        7: "مهر",
        8: "آبان",
        9: "آذر",
        10: "دی",
        11: "بهمن",
        12: "اسفند",
    }

    return f"{month_names[month]} {year}"


df["created_at_shamsi"] = df["created_at_month"].apply(gregorian_month_to_shamsi)
df["created_at_shamsi_readable"] = df["created_at_shamsi"].apply(shamsi_month_readable)


# این ستون‌ها در ادامه پروژه استفاده نمی‌شوند یا مقدارهایشان برای تحلیل قابل اتکا نیست
cols_to_drop = [
    "rent_to_single",
    "cost_per_extra_person",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
    "rent_price_on_regular_days",
    "extra_person_capacity",
    "has_water",
    "has_electricity",
    "has_gas",
]

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])


print("Step 3.2 completed.")
print("Shape:", df.shape)

df[[
    "created_at_month",
    "created_at_shamsi",
    "created_at_shamsi_readable",
]].head()

<div dir="rtl" align="right">

## 4. ساخت ستون‌های قیمت و متراژ

در این بخش ستون‌های کمکی مربوط به قیمت، اجاره، رهن و متراژ ساخته می‌شوند.

هدف این مرحله آماده‌سازی داده برای تحلیل قیمت، محاسبه قیمت واحد و مدل‌سازی است.

ستون‌های مهم این مرحله شامل موارد زیر هستند:

`transaction_type`

`price_value_pos`

`rent_value_pos`

`credit_value_pos`

`area_for_unit_price`

`sale_price_per_m2`

`monthly_rent_equivalent_per_m2`

</div>

<div dir="rtl" align="right">

## 5. ساخت قیمت هدف و flagهای اعتبارسنجی قیمت

در این بخش قیمت هدف برای هر آگهی ساخته می‌شود.

همچنین flagهایی برای تشخیص رکوردهای مناسب تحلیل قیمت ایجاد می‌شوند.
در این مرحله رکوردی حذف نمی‌شود و فقط ستون‌های کمکی و flag اضافه می‌شوند.

ستون‌های مهم این مرحله شامل موارد زیر هستند:

`target_price`

`target_price_type`

`target_price_per_m2`

`is_valid_for_price_analysis`

خروجی این مرحله:

`cleaned_step2_benyamin_v2.parquet`

</div>

<div dir="rtl" align="right">

## 6. پاکسازی داده‌های جغرافیایی

در این بخش وضعیت مختصات جغرافیایی بررسی می‌شود.

با استفاده از ستون‌های اولیه‌ی جغرافیایی، یک flag نهایی ساخته می‌شود تا مشخص کند کدام رکوردها برای تحلیل مکانی قابل استفاده هستند.

ستون مهم این مرحله:

`is_valid_geo_for_analysis`

منطق این ستون:

* مقدار `True` یعنی رکورد مختصات معتبر دارد.
* مقدار `False` یعنی رکورد مختصات ندارد یا مختصات آن نامعتبر است.

</div>

<div dir="rtl" align="right">

## 7. تبدیل مختصات به UTM

در این بخش مختصات `latitude` و `longitude` برای رکوردهای معتبر به مختصات UTM تبدیل می‌شوند.

مختصات UTM برحسب متر هستند و برای تحلیل مکانی، محاسبه فاصله، نقشه و clustering مناسب‌تر از مختصات طول و عرض جغرافیایی هستند.

ستون‌های ساخته‌شده در این مرحله:

`utm_easting`

`utm_northing`

`utm_zone_number`

`utm_zone_letter`

نکته مهم: داده‌ها ممکن است در چند UTM zone مختلف قرار داشته باشند؛ بنابراین در مراحل بعدی، مخصوصاً clustering سراسری، باید به چند-zone بودن مختصات توجه شود.

</div>

<div dir="rtl" align="right">

## 8. ذخیره خروجی نهایی preprocessing

در این بخش دیتاست نهایی preprocessing ذخیره می‌شود.

خروجی نهایی این نوت‌بوک:

`cleaned_step3_geo.parquet`

این فایل برای مراحل بعدی پروژه استفاده می‌شود:

* آمار توصیفی
* نقشه و تحلیل مکانی
* آزمون فرض
* خوشه‌بندی
* پیش‌بینی قیمت

</div>

<div dir="rtl" align="right">

## 9. بررسی نهایی خروجی

در انتهای نوت‌بوک، شکل دیتاست، ستون‌های جدید و تعداد مقادیر گم‌شده‌ی مهم بررسی می‌شود تا مطمئن شویم خروجی نهایی درست ساخته شده است.

</div>